## **Orienting Wayfair Products in Semantic Vector-Space for Analytics and Search**

### Project Overview

The axiom of the information age is that **data** makes businesses winners. This tells an incomplete story: data acquisition is all well and good, but the generation of actionable **insight** from said data is what creates value for companies. The project proposes a method for drawing further insight from existing Wayfair data collections – the orientation of Wayfair products in semantic vector-space using customer product reviews.

How do you quantify something as subjective as the relative 'comfortableness' of one sofa to another? Would being able to rank different sofas by comfort help Wayfair decide what products to host and recommend to customers? Would these rankings inform consumer purchase decisions, perhaps by allowing shoppers to search not by keyword, but by a desired quality?

In this project, I apply the tokenization-vectorization paradigm integral in Natural Language Processing workflows to product reviews available on Wayfair's website in order to generate a method of ranking different sofas based on the desired attribute of comfort.

### Workflow Design

We will begin by defining a workflow that we can later apply to public Wayfair product reviews.

Assume, if you will, that there exists a vector representation of the word 'comfortable'. This vector represents the ideal that all sofas are striving towards – trying to get as close to being universally comfortable as they possibly can. We want to create a way to compare how close different sofas are able to get to this ideal.

We will consider the identity of a sofa to be the sum of its product reviews; that is, the esssences of a given sofa is the distillation of all the reviews left by consumers who have tried the product and reported on their experience. We want to position these identities within the same vector-space as the ideal of 'comfortable' and compare the relative distances of each sofa from that ideal. We can then generate some sort of ranking for which sofas are the most comfortable and which sofas are the least.

Our workflow is as follows: we will **tokenize** each of the reviews for a given product, generate **token embeddings**, pool these into **sentence/review embeddings**, pool these into holistic **product embeddings**, and then calculate **cosine similarity** between the product embedding and the benchmark ideal. If we do this for multiple products, we can compare their cosine similarity values and determine which falls closest to the ideal and which falls farthest from it, developing a **ranking**.

First, we need to decide on a tokenizer/model to use to generate the embeddings, as well as a pooling method. We will make these decisions by examining how different choices affect the results for a synthetic textset.

We'll begin by importing some relevant libraries.

In [1]:
# IMPORTS
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd

We will define two different options for pooling token embeddings to generate sentence/review embeddings: mean pooling and CLS pooling. Additionally, we will define a method for pooling sentence/review embeddings into full product embeddings.

*The code for mean_pooling was taken from the model card for the ['all-mpnet-base-v2'](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) transfomer model and modified slightly.*

In [2]:
# POOLING FUNCTIONS

# mean pooling: token embeddings -> sentence/review embeddings
def mean_pooling(token_embeddings, attention_mask):
    attention_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * attention_mask, 1) / torch.clamp(attention_mask.sum(1), min=1e-9)

# CLS pooling: token embeddings -> sentence/review embeddings
def cls_pooling(token_embeddings, attention_mask):
    for i in range(attention_mask.shape[0]):
        for j in range(1, attention_mask.shape[1]):
            attention_mask[i][j] = 0
    return mean_pooling(token_embeddings, attention_mask)

# product pooling: review embeddings -> product embedding
def prod_pooling(input_embeddings):
    return torch.sum(input_embeddings, 0) / input_embeddings.shape[0]

It's worth noting that the prod_pooling method weights all input embeddings equally – in other words, no review is more or less important than any other.

Next, we will define a couple of different tokenizer/model options, all of them available from Hugging Face.

[sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2)
[sentence-transformers/nli-mpnet-base-v2](https://huggingface.co/sentence-transformers/nli-mpnet-base-v2)
[princeton-nlp/sup-simcse-bert-base-uncased](https://huggingface.co/princeton-nlp/sup-simcse-bert-base-uncased)

In [3]:
# DEFINE TOKENIZER + MODEL

# model_to_use = 'sentence-transformers/all-mpnet-base-v2'
# model_to_use = 'sentence-transformers/nli-mpnet-base-v2'
model_to_use = 'princeton-nlp/sup-simcse-bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_to_use)
model = AutoModel.from_pretrained(model_to_use)

As I noted above, there are two decisions we need to make before we can start working with Wayfair data – which tokenizer/model option to use, and which pooling method to use. To help me decide, I experimented with different models and pooling methods on a synthetic textset, trying to see how the position of the embeddings produced changed with the choices I made.

In [4]:
# CREATE SYNTHETIC TEXTSET
syn = ["comfortable", \
       "This is the most comfortable sofa ever.", \
       "This is the most uncomfortable sofa ever.", \
       "comfortable", \
       "very comfortable", \
       "Loved this sofa, definitely a good one.", \
       "Hated this sofa, comfortable but should not buy because the color is obscene.", \
       "Cozy and nice.", \
       "Cozy but would not buy."]

For each entry in the synthetic textset, I generated a sentence-level embedding and calculated the cosine similarity between each entry and all of the others.

In [5]:
# RUN TRIAL WORKFLOW ON SYNTHETIC TEXTSET

# tokenize syn
token_syn = tokenizer(syn, padding=True, truncation=True, return_tensors='pt')

# get embeddings for tokens in each syn entry
with torch.no_grad():
    model_out_syn = model(**token_syn)

# pool token embeddings to get sentence-level embeddings for each entry in syn, L2 norm
# sentence_syn = F.normalize(mean_pooling(model_out_syn[0], token_syn.attention_mask), p=2, dim=1)
sentence_syn = F.normalize(cls_pooling(model_out_syn[0], token_syn.attention_mask), p=2, dim=1)

# display each cosine similarity as a DataFrame
sentence_1 = []
sentence_2 = []
cos_sim = []
seen = []
for i in range(len(syn)):
    for j in range(i + 1, len(syn)):
        if(not((syn[i], syn[j]) in seen or (syn[j], syn[i]) in seen)):
            sentence_1.append(syn[i])
            sentence_2.append(syn[j])
            cos_sim.append((sentence_syn[i] @ sentence_syn[j]).item())
            seen.append((syn[i], syn[j]))
            # print(f'({syn[i]} / {syn[j]}) = {(sentence_syn[i] @ sentence_syn[j]).item()}')
results_syn = pd.DataFrame({'sentence_1': sentence_1, 'sentence_2': sentence_2, 'cos_sim': cos_sim})
display(results_syn)

,sentence_1,sentence_2,cos_sim
0,comfortable,This is the most comfortable sofa ever.,0.661939
1,comfortable,This is the most uncomfortable sofa ever.,0.543853
2,comfortable,comfortable,1.000000
3,comfortable,very comfortable,0.975020
4,comfortable,"Loved this sofa, definitely a good one.",0.677291
5,comfortable,"Hated this sofa, comfortable but should not bu...",0.508968
6,comfortable,Cozy and nice.,0.900629
7,comfortable,Cozy but would not buy.,0.667207
8,This is the most comfortable sofa ever.,This is the most uncomfortable sofa ever.,0.836338
9,This is the most comfortable sofa ever.,very comfortable,0.718477


Holding the pooling strategy constant at mean_pooling, I ran this workflow for each of my tokenizer/model options and got the following results:

In [6]:
all_models = pd.read_csv('all-models.csv')
display(all_models)

,sentence_1,sentence_2,cos_sim_all_mpnet,cos_sim_nli_mpnet,cos_sim_sup_simcse
0,comfortable,This is the most comfortable sofa ever.,0.516279,0.325693,0.617525
1,comfortable,This is the most uncomfortable sofa ever.,0.455039,0.220350,0.473663
2,comfortable,comfortable,1.000000,1.000000,1.000000
3,comfortable,very comfortable,0.839744,0.774429,0.938370
4,comfortable,"Loved this sofa, definitely a good one.",0.352047,0.333620,0.627388
5,comfortable,"Hated this sofa, comfortable but should not bu...",0.385699,0.113662,0.436114
6,comfortable,Cozy and nice.,0.657779,0.568009,0.852882
7,comfortable,Cozy but would not buy.,0.514626,0.173432,0.611740
8,This is the most comfortable sofa ever.,This is the most uncomfortable sofa ever.,0.793414,0.677423,0.811101
9,This is the most comfortable sofa ever.,very comfortable,0.567336,0.538466,0.697867


In reviewing the data, I noted that while nli-mpnet did poorly when it same to entries with similar semantic meaning ('comfortable' vs. 'This is the most comfortable sofa ever.' had a low cosine similarity of 0.325). However, all-mpnet and sup-simcse both did poorly when it came to entries with contrastic semantic meaning ('This is the most comfortable sofa ever.' vs. 'This is the most uncomfortable sofa ever.' saw high cosine similarities of 0.793 and 0.811).

For the purposes of this project, I ended up choosing sup-simcse, reasoning that larger models, which were out-of-scope for this proof-of-concept given my resource constraints, might have better contrastive recognition. I prioritized instead recognition of semantic similarity. I chose sup-simcse over all-mpnet because it had higher cosine similarities for semantically-similar pairings like ('comfortable', 'This is the most comfortable sofa ever.') and ('comfortable', 'very comfortable').

I ran similar trials for mean_pooling vs. cls_pooling, holding the tokenizer/model steady (sup-simcse):

In [7]:
all_pool = pd.read_csv('all-pool.csv')
display(all_pool)

,sentence_1,sentence_2,cos_sim_mean_pool,cos_sim_cls_pool
0,comfortable,This is the most comfortable sofa ever.,0.617525,0.661939
1,comfortable,This is the most uncomfortable sofa ever.,0.473663,0.543853
2,comfortable,comfortable,1.000000,1.000000
3,comfortable,very comfortable,0.938370,0.975020
4,comfortable,"Loved this sofa, definitely a good one.",0.627388,0.677291
5,comfortable,"Hated this sofa, comfortable but should not bu...",0.436114,0.508968
6,comfortable,Cozy and nice.,0.852882,0.900629
7,comfortable,Cozy but would not buy.,0.611740,0.667207
8,This is the most comfortable sofa ever.,This is the most uncomfortable sofa ever.,0.811101,0.836338
9,This is the most comfortable sofa ever.,very comfortable,0.697867,0.718477


Overall, it seemed as though the CLS pooling metrics were higher across the board than the mean pooling ones. Because I chose sup-simcse as my model, which trains the CLS (actually the \<s>) token on holistic semantic meaning across the entire sentence anyway, I decided to go with CLS pooling as my method of choice.

Now that these design decisions have been made, I can go ahead and run the workflow on data from Wayfair.

### Ranking Wayfair Products

I manually aggregated publicly-available customer reviews for three products from Wayfair's website. The metadata is as follows:

The data was collected on Thursday, January 15, 2026. A search was made for 'comfortable sofa', and the results were sorted by 'Customer Rating'. It is worth noting that the results displayed were not replicable in a separate attempt using the same parameters.

The reviews were sorted by 'Latest' and only ones with the 'Verified Buyer' tag were considered. The reviews had to have some English text to be included. Product information that appeared by template in some of the reviews was excluded. In this way, the top 20 customer reviews were collected for each of three products.

The data is displayed below:

In [8]:
wayfair = pd.read_csv('wayfair.csv')
display(wayfair)

,sku,review_text
0,W008103065,This sofa is so beautiful and comfortable. I’m...
1,W008103065,This is a hit in our media room. It is comfort...
2,W008103065,I've had this couch for roughly one year and I...
3,W008103065,"Obsessed with this couch, I live on it, but wh..."
4,W008103065,Purchased quite a while back. Still in box as ...
5,W008103065,The sofa is very easy to assemble with clear i...
6,W008103065,My favourite couch I have ever owned. Love it-...
7,W008103065,Love this couch! Exactly what we were looking ...
8,W008103065,Perfect and very comfortable
9,W008103065,"Better than I expected, the back cushions are ..."


To begin, I will separate the reviews by product and create a benchmark against which the product embeddings for each sofa will be compared. This benchmark will contain 20 instances of the word 'comfortable' and will be processed (tokenized, vectorized, and pooled) just as the customer reviews for the products are.

In [9]:
bench = ['comfortable'] * 20
W008 = wayfair.loc[wayfair.sku == 'W008103065', 'review_text'].to_list()
W011 = wayfair.loc[wayfair.sku == 'W011131433', 'review_text'].to_list()
W000 = wayfair.loc[wayfair.sku == 'W000902139', 'review_text'].to_list()

Now, let's apply the full workflow to each of these textsets:

In [10]:
transform = {'bench': bench, 'W008': W008, 'W011': W011, 'W000': W000}
for item in transform:
    # tokenize
    transform[item] = tokenizer(transform[item], padding=True, truncation=True, return_tensors='pt')
    temp = transform[item]
    # get token embeddings
    with torch.no_grad():
        transform[item] = model(**transform[item])
    # pool sentence-level, L2 norm
    transform[item] = F.normalize(cls_pooling(transform[item][0], temp.attention_mask), p=2, dim=1)
    # pool product-level, L2 norm
    transform[item] = F.normalize(prod_pooling(transform[item]), p=2, dim=0)
# calculate and display cosine similarities
rank = []
for item in transform:
    cos_sim = (transform['bench'] @ transform[item]).item()
    print(f'bench vs. {item} = {round(cos_sim, 5)}')
    rank.append((cos_sim, item))

bench vs. bench = 1.0
bench vs. W008 = 0.80897
bench vs. W011 = 0.80592
bench vs. W000 = 0.81372


Using these calculated cosine similarities, we can rank how close each of the three sofas is to the benchmark (greatest to least).

In [11]:
rank.sort()
print('Most Comfortable')
counter = len(rank) - 2
while(counter >= 0):
    print(rank[counter][1])
    counter -= 1
print('Least Comfortable')

Most Comfortable
W000
W008
W011
Least Comfortable


### Discussion

This project represents the application of an existing paradigm, that of the NLP workflow, to a different context – here, we situate the product itself, not just the text reviews, in vector-space as a means of comparing distance to some benchmark ideal. The proposal has tremendous potential in terms of value added to Wayfair operations. Leveraging the above workflow would allow Wayfair to source insight directly from customers (data that is already collected and stored) and compare product performance for a desired attribute across their entire volume of merchandise. This insight could inform a number of decisions – what products to stock in Wayfair's new physical stores, how to present products to digital shoppers, which products should be added to the catalog. Ultimately, this proposal seeks to maximize positive experiences shopping at Wayfair.

Incorporating these insights into a search mechanism could allow users to find products based on a desired attribute rather than by keywork. Importantly, because the ranking is based on user-sourced reviews, customers can have faith that the product they are browsing have been vetted by previous purchasers as achieving whatever the desired attribute might be.

There are limitations of this project as I have conducted it, specifically computational resource and data volume constraints. Ideally, larger models optimized for contrastive semantic recognition might perform better in terms of the embeddings produced. Furthermore, I opted not to webscrape data from Wayfair's page – the manual aggregation of the reviews limited the volume of data that went into the product embedding. In practice, more products could be examined, and more reviews per product would be included in the textset.

I will end with a discussion of some concerns worth noting (this is not an exhaustive list). There is a danger of systemic bias infiltrating result – if there is a systemic reason why certain products are described certain ways, this could unfairly bias the ranking and disadvantage certain products from being sold or seen by consumers. Additionally, products with relatively few reviews to their name might be unfairly advantaged or disadvantaged. The weight of each review matters more with a smaller sample size – if the first couple of reviews for a product happen to be negative, this could lead to falling purchase rates for that product, which in turn results in falling review rates and a narrower chance at redemption. In this way, the application of these ideas to Wayfair systems without guardrails could unfairly disadvantage certain products from being sold or seen. Another potential issue arises when considering how different natural language might differentially impact a product's placement in vector-space - different language might not be applied equitably in terms of moving a product closer or farther from the the ideal of 'comfortable'.

Generally speaking, the problem with rankings is that if a product is no longer sold or prominently presented to consumers because of its ranking, it doesn't have a fair chance at ameliorating that ranking, and it will be systematically kept from consumers. This is a concern given vendors selling through Wayfair depend on user exposure to their products. This is a huge issue that requires careful consideration and extension of the ideas presented before any such system can be discussed for integration into Wayfair operations.